# RAG: Baseline Retrieval vs. Cross-Encoder Re-Ranking

Two retrieval pipelines run over the **same chunked corpus** and are compared:

- **Pipeline 1 — Baseline:** dense bi-encoder retrieval. Embed the query, pull the top 10 chunks by cosine similarity, hand the top 3 to the LLM.
- **Pipeline 2 — Re-ranking:** take the same top 10, re-score every `(query, chunk)` pair with a cross-encoder, re-sort, and hand the new top 3 to the LLM.

The corpus is the Catan base-game rulebook (rules + almanac) — dense with exceptions, penalties and role limits, which makes retrieval mistakes easy to see.

## 1. Setup & Environment

Models used:
- Bi-encoder: `sentence-transformers/all-MiniLM-L6-v2`
- Cross-encoder: `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Answer generation: `Qwen/Qwen2.5-1.5B-Instruct`, run locally with Hugging Face `transformers`

All three models download from Hugging Face the first time they are used and then run on the local CPU. No API key or network service is required.

In [ ]:
import os, re, time, textwrap, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pypdf
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# Retrieval settings
BI_ENCODER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_K = 10   # candidates pulled by the baseline
TOP_N = 3    # chunks actually sent to the LLM

# Chunking settings
CHUNK_CHARS = 700
CHUNK_OVERLAP = 120

# Answer generation (local Hugging Face model)
LOCAL_MODEL = os.environ.get("LOCAL_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
print("Answer model (local):", LOCAL_MODEL)

## 2. Dataset

A single complex PDF is the knowledge base. The notebook downloads it on first run and keeps it in `data/`.

Each page is extracted separately so every chunk can be traced back to a page number.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PDF_PATH = DATA_DIR / "catan_rules.pdf"
PDF_URL = "https://world-of-board-games.com.sg/docs/Catan-5ed.pdf"

if not PDF_PATH.exists():
    print("Downloading corpus ...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
print("PDF size (KB):", PDF_PATH.stat().st_size // 1024)


def load_pages(path):
    reader = pypdf.PdfReader(str(path))
    if reader.is_encrypted:
        reader.decrypt("")
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if text:
            pages.append({"page": i + 1, "text": text})
    return pages


pages = load_pages(PDF_PATH)
print(f"Loaded {len(pages)} non-empty pages")
print(pages[1]["text"][:400])

## 3. Chunking

The pages are split into overlapping character windows. Paragraph boundaries are respected where possible so a rule and its exception tend to stay together, and a small overlap keeps context from being cut mid-sentence. Each chunk records its source page.

In [ ]:
def chunk_pages(pages, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    chunks = []
    for item in pages:
        page_no, text = item["page"], item["text"]
        paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
        buf = ""
        for para in paragraphs:
            if len(buf) + len(para) + 1 <= size:
                buf = (buf + " " + para).strip()
            else:
                if buf:
                    chunks.append({"page": page_no, "text": buf})
                if len(para) > size:
                    # a single long paragraph: slide a window across it
                    for start in range(0, len(para), size - overlap):
                        chunks.append({"page": page_no, "text": para[start:start + size]})
                    buf = ""
                else:
                    tail = buf[-overlap:] if buf else ""
                    buf = (tail + " " + para).strip()
        if buf:
            chunks.append({"page": page_no, "text": buf})
    return chunks


chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(t) for t in chunk_texts) // len(chunk_texts)} chars")
print("\nExample chunk:\n", chunk_texts[5])

## 4. Pipeline 1 — Baseline Retrieval (Bi-Encoder)

The bi-encoder embeds each chunk independently into a vector. Vectors are L2-normalised and stored in a FAISS inner-product index, so an inner-product search is equivalent to cosine similarity. For a query we embed it the same way and pull the `TOP_K` nearest chunks.

In [ ]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

chunk_embeddings = bi_encoder.encode(
    chunk_texts, normalize_embeddings=True, show_progress_bar=True
).astype("float32")

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)
print(f"Indexed {index.ntotal} chunks, dimension {dim}")


def baseline_retrieve(query, k=TOP_K):
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{"id": int(i), "score": float(s)} for i, s in zip(ids[0], scores[0])]

## 5. Pipeline 2 — Re-Ranking (Cross-Encoder)

The cross-encoder takes the `TOP_K` baseline candidates and reads each `(query, chunk)` pair **together** in one pass, so its score reflects direct interaction between the query and the passage rather than two separate embeddings. We re-sort the 10 candidates by this score and keep the new top 3.

In [ ]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)


def rerank(query, candidates):
    ids = [c["id"] for c in candidates]
    pairs = [(query, chunk_texts[i]) for i in ids]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(
        ({"id": i, "score": float(s)} for i, s in zip(ids, scores)),
        key=lambda x: x["score"], reverse=True,
    )
    return ranked